In [1]:
from torch.utils.data import DataLoader
from datasets import load_from_disk
from src.data_utils import generate_ravel_dataset, get_ravel_collate_fn, filter_dataset

from transformers import AutoTokenizer

%load_ext autoreload
%autoreload 2

In [2]:
tokenizer = AutoTokenizer.from_pretrained("/work/frink/models/llama3-8B-HF")
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

train_dataset = load_from_disk("./data/ravel/mixed_train")
test_dataset = load_from_disk("./data/ravel/mixed_test")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
all_train_entities = set()
all_test_entities = set()
for data in train_dataset:
    all_train_entities.add(data["entity"])
    
for data in test_dataset:
    all_test_entities.add(data["entity"])
    
print(len(all_train_entities), len(all_test_entities))

1785 1302


In [4]:
from src.llama3.model import RavelInterpretorHypernetwork

hypernetwork = RavelInterpretorHypernetwork(
    model_name_or_path="/work/frink/models/llama3-8B-HF",
    num_editing_heads=32,
    intervention_layer=15,
    das_intervention=True,
    das_dimension=128,
    allow_selective_column_space=True
)
hypernetwork = hypernetwork.to("cuda")
hypernetwork.load_model("./models/selective_das/final_model")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
collate_fn = get_ravel_collate_fn(tokenizer, add_space_before_target=True)
dataloader = DataLoader(test_dataset, batch_size=8, collate_fn=collate_fn, shuffle=False)

In [29]:
hypernetwork.eval_accuracy(dataloader, inference_mode=None, eval_n_label_tokens=3)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


({'causal': 0.039473684210526314,
  'isolate': 0.9736842105263158,
  'disentangle': 0.506578947368421},
 3.410506588063742,
 [12,
  25,
  62,
  96,
  97,
  133,
  168,
  170,
  192,
  221,
  235,
  293,
  303,
  333,
  335,
  340,
  417,
  431,
  446,
  511,
  526,
  532,
  533,
  534,
  535,
  536,
  537,
  538,
  539,
  540,
  541,
  542,
  543,
  544,
  545,
  546,
  547,
  548,
  549,
  550,
  551,
  552,
  553,
  554,
  555,
  556,
  557,
  558,
  559,
  560,
  561,
  562,
  563,
  564,
  565,
  566,
  567,
  568,
  569,
  570,
  571,
  572,
  573,
  574,
  575,
  576,
  577,
  578,
  579,
  580,
  581,
  582,
  583,
  584,
  585,
  586,
  587,
  588,
  589,
  590,
  591,
  592,
  593,
  594,
  595,
  596,
  597,
  598,
  599,
  600,
  601,
  602,
  603,
  604,
  605,
  606,
  607,
  608,
  609,
  610,
  611,
  612,
  613,
  614,
  615,
  616,
  617,
  618,
  619,
  620,
  621,
  623,
  624,
  625,
  626,
  627,
  628,
  629,
  630,
  631,
  632,
  633,
  634,
  635,
  636,
  637,

In [ ]:
hypernetwork.interpretor.bidding_threshold = 0.2
plot, _ = hypernetwork.plot_heatmap(dataloader, idxs=1300, inference_mode="bidding_argmax", annot=True)

NameError: name 'hypernetwork' is not defined

In [12]:
hypernetwork.interpretor.das_module.temperature

Parameter containing:
tensor(0.1001, device='cuda:0', dtype=torch.bfloat16)